In [9]:
import os
import re
import json
import uuid
import hashlib
import time
from typing import Literal
from collections import Counter
from dotenv import load_dotenv
from pydantic import BaseModel
from tqdm import tqdm

from groq import Groq
import cohere
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocItemLabel

load_dotenv()

# Clients
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
cohere_client = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("✓ Groq client ready")
print("✓ Cohere client ready")
print("✓ Environment loaded")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4940.46it/s]


✓ Groq client ready
✓ Cohere client ready
✓ Environment loaded


Step 2 : Load artifacts
 

In [3]:
# Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("financial-reconciliation-v4")

# Docstore
with open("../data/processed/docstore.json", "r") as f:
    docstore = json.load(f)

# BM25
bm25_encoder = BM25Encoder()
bm25_encoder.load("../data/processed/bm25_encoder.json")



print(f"✓ Pinecone connected: financial-reconciliation-v4")
print(f"✓ Docstore: {len(docstore)} parents")
print(f"✓ BM25 loaded")
print(f"✓ SentenceTransformer loaded")
print(f"\nIndex stats:")
print(index.describe_index_stats())

✓ Pinecone connected: financial-reconciliation-v4
✓ Docstore: 656 parents
✓ BM25 loaded
✓ SentenceTransformer loaded

Index stats:
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL_PC': {'vector_count': 435},
                'MSFT_PC': {'vector_count': 1001}},
 'total_vector_count': 1436,
 'vector_type': 'dense'}


Step 3 : Helper functions and Query Router

In [4]:
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]
EMBEDDING_DIM = 384


def scale_dense(vector: list, alpha: float) -> list:
    return [v * alpha for v in vector]


def scale_sparse(vector: dict, alpha: float) -> dict:
    return {
        "indices": vector["indices"],
        "values": [v * (1 - alpha) for v in vector["values"]]
    }


class QueryRoute(BaseModel):
    chunk_type: Literal["table", "prose"]
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "MD&A", "Risk Factors", "Legal Proceedings",
        "Notes", "Shareholders Equity", "Any"
    ]


def route_query(query: str) -> dict:
    system_prompt = """You are a financial document query router for SEC 10-Q filings.

PRIORITY RULES:
- inventory/inventories → Balance Sheet, table
- dividends PAID as cash outflow → Cash Flow, table
- R&D as percentage or ratio → MD&A, prose
- EPS computation or weighted average shares → Notes, table
- gross margin as dollar figure → Income Statement, table
- R&D dollar amount → Income Statement, table
- change/grew/increased about a metric → MD&A, prose
- share repurchase program details → Notes, prose

SECTION GUIDE:
- Income Statement: revenue, net sales, gross margin, operating income, net income, cost of sales
- Balance Sheet: total assets, cash, liabilities, equity, receivables, inventory
- Cash Flow: operating activities, investing, financing, dividends paid, capex
- MD&A: why metrics changed, percentage change, growth reasons, segment performance
- Risk Factors: risks, uncertainties, challenges
- Legal Proceedings: lawsuits, investigations
- Notes: RSUs, debt details, EPS computation, weighted average shares, share repurchase
- Shareholders Equity: retained earnings, AOCI
- Any: spans multiple sections

CHUNK TYPE: NUMBER/FIGURE → table | EXPLANATION/WHY/HOW → prose

Return ONLY valid JSON: {"chunk_type": "table", "section": "Income Statement"}"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    try:
        result = json.loads(response.choices[0].message.content)
        route = QueryRoute(**result)
        return {"chunk_type": route.chunk_type, "section": route.section}
    except:
        return {"chunk_type": "prose", "section": "Any"}


def retrieve_with_parent_child(
    query: str,
    ticker: str,
    section: str = None,
    chunk_type: str = None,
    top_k: int = 5,
    alpha: float = 0.5
) -> list:

    namespace = f"{ticker}_PC"
    dense_vector = embedding_model.encode(
        query, normalize_embeddings=True
    ).tolist()
    sparse_vector = bm25_encoder.encode_queries(query)

    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section and section != "Any":
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}

    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k * 3,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )

    seen_parents = set()
    retrieved = []

    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        if not parent_id or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {**match.metadata, "parent_id": parent_id}
            })
        if len(retrieved) >= top_k:
            break

    return retrieved


def retrieve_with_router(query: str, ticker: str, top_k: int = 5, alpha: float = 0.5):
    route = route_query(query)
    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=route["section"],
        chunk_type=route["chunk_type"],
        top_k=top_k,
        alpha=alpha
    )
    return retrieved, route


print("✓ scale_dense() + scale_sparse()")
print("✓ route_query() — Groq llama-3.3-70b")
print("✓ retrieve_with_parent_child()")
print("✓ retrieve_with_router()")

✓ scale_dense() + scale_sparse()
✓ route_query() — Groq llama-3.3-70b
✓ retrieve_with_parent_child()
✓ retrieve_with_router()


Step 4 : Basic QA function

In [10]:
SYSTEM_PROMPT = """You are a financial analyst assistant specializing in SEC filings.

Answer questions based ONLY on the provided financial document context.
Rules:
- State numbers exactly as they appear in the document
- Financial figures in 10-Q tables are in millions unless stated otherwise
- For Microsoft Q1 FY2026 → look for "Three Months Ended December 31, 2025"
- For Apple Q2 FY2026 → look for "Three Months Ended March 28, 2026"
- The FIRST data column after the header is usually the most recent period
- Always append "million" to dollar figures
- Always state which period you are reporting
- If you cannot find the answer say: "Not found in the provided filing sections"
- Never fabricate numbers

CRITICAL: When table has columns "2025 | 2024" under "Three Months Ended December 31":
  - 2025 column = Q1 FY2026 (most recent)
  - 2024 column = Q1 FY2025 (prior year)"""


def answer_question(query: str, ticker: str, top_k: int = 3, alpha: float = 0.5) -> dict:
    start_time = time.time()

    # Step 1 — Route
    route = route_query(query)

    # Step 2 — Retrieve
    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=route["section"],
        chunk_type=route["chunk_type"],
        top_k=top_k,
        alpha=alpha
    )

    # Step 3 — Build context
    context = "\n\n---\n\n".join([
        f"[{r['metadata'].get('section')} | Page {r['metadata'].get('page')} | {r['metadata'].get('chunk_type')}]\n{r['parent_content']}"
        for r in retrieved
    ])

    # Step 4 — Generate
    user_prompt = f"Financial filing context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    answer = response.choices[0].message.content
    total_time = time.time() - start_time

    return {
        "query": query,
        "ticker": ticker,
        "answer": answer,
        "route": route,
        "sources": [
            {
                "section": r["metadata"].get("section"),
                "page": r["metadata"].get("page"),
                "chunk_type": r["metadata"].get("chunk_type"),
                "child_score": r["child_score"]
            }
            for r in retrieved
        ],
        "usage": {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        },
        "latency_ms": round(total_time * 1000)
    }


# Test
result = answer_question("What was Apple total net sales in Q2 2026?", "AAPL")

print(f"Query  : {result['query']}")
print(f"Answer : {result['answer']}")
print(f"Route  : {result['route']}")
print(f"Latency: {result['latency_ms']}ms")
print(f"Tokens : {result['usage']}")
print(f"\nSources:")
for s in result['sources']:
    print(f"  - {s['section']} | page {s['page']} | score {s['child_score']:.4f}")

Query  : What was Apple total net sales in Q2 2026?
Answer : According to the provided filing context, under the "Income Statement" table on Page 4.0, the total net sales for the "Three Months Ended March 28, 2026" is $111,184 million.
Route  : {'chunk_type': 'table', 'section': 'Income Statement'}
Latency: 2645ms
Tokens : {'input_tokens': 1509, 'output_tokens': 47, 'total_tokens': 1556}

Sources:
  - Income Statement | page 4.0 | score 0.1862
  - Income Statement | page 5.0 | score 0.1548


In [16]:
def answer_question_traced(query: str, ticker: str, top_k: int = 3, alpha: float = 0.5) -> dict:
    start_time = time.time()

    with langfuse.start_as_current_observation(
        as_type="span",
        name="answer_question",
        input={"query": query, "ticker": ticker}
    ) as root_span:

        # Step 1 — Route
        with langfuse.start_as_current_observation(
            as_type="span",
            name="route_query",
            input={"query": query}
        ) as route_span:
            route = route_query(query)
            route_span.update(output=route)

        # Step 2 — Retrieve
        with langfuse.start_as_current_observation(
            as_type="span",
            name="retrieve_chunks",
            input={"ticker": ticker, **route}
        ) as retrieve_span:
            retrieved = retrieve_with_parent_child(
                query=query,
                ticker=ticker,
                section=route["section"],
                chunk_type=route["chunk_type"],
                top_k=top_k,
                alpha=alpha
            )
            retrieve_span.update(output={
                "chunks_retrieved": len(retrieved),
                "sections": [r["metadata"].get("section") for r in retrieved]
            })

        # Step 3 — Build context
        context = "\n\n---\n\n".join([
            f"[{r['metadata'].get('section')} | Page {r['metadata'].get('page')}]\n{r['parent_content']}"
            for r in retrieved
        ])

        # Step 4 — Fetch prompt from Langfuse
        try:
            prompt_obj = langfuse.get_prompt(
                "financial_analyst_system",
                label="production"
            )
            system_prompt = prompt_obj.prompt
            prompt_version = prompt_obj.version
        except:
            system_prompt = SYSTEM_PROMPT
            prompt_version = 0

        user_prompt = f"Financial filing context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

        # Step 5 — Generate
        with langfuse.start_as_current_observation(
            as_type="generation",
            name="llm_generation",
            model="llama-3.1-8b-instant",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        ) as gen_span:
            response = groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0
            )
            answer = response.choices[0].message.content
            gen_span.update(
                output=answer,
                usage_details={
                    "input": response.usage.prompt_tokens,
                    "output": response.usage.completion_tokens
                }
            )

        total_time = time.time() - start_time
        root_span.update(output={"answer": answer})

    langfuse.flush()

    return {
        "query": query,
        "ticker": ticker,
        "answer": answer,
        "route": route,
        "prompt_version": prompt_version,
        "sources": [
            {
                "section": r["metadata"].get("section"),
                "page": r["metadata"].get("page"),
                "chunk_type": r["metadata"].get("chunk_type"),
                "child_score": r["child_score"]
            }
            for r in retrieved
        ],
        "usage": {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        },
        "latency_ms": round(total_time * 1000)
    }


# Test
result = answer_question_traced(
    "What was Apple total net sales in Q2 2026?",
    "AAPL"
)

print(f"Query         : {result['query']}")
print(f"Answer        : {result['answer']}")
print(f"Prompt version: {result['prompt_version']}")
print(f"Latency       : {result['latency_ms']}ms")
print(f"Tokens        : {result['usage']}")
print(f"\nCheck Langfuse dashboard → Tracing")

Query         : What was Apple total net sales in Q2 2026?
Answer        : Three Months Ended March 28, 2026, Apple total net sales was $111,184 million.
Prompt version: 4
Latency       : 2644ms
Tokens        : {'input_tokens': 1527, 'output_tokens': 23, 'total_tokens': 1550}

Check Langfuse dashboard → Tracing


In [17]:
test_queries = [
    ("What was Apple total net sales in Q2 2026?", "AAPL"),
    ("What was Apple net income in Q2 2026?", "AAPL"),
    ("What was Apple gross margin in Q2 2026?", "AAPL"),
    ("Why did Apple iPhone revenue increase in Q2 2026?", "AAPL"),
    ("What was Apple cash and cash equivalents?", "AAPL"),
    ("What was Apple operating income in Q2 2026?", "AAPL"),
    ("What was Microsoft net income in Q1 FY2026?", "MSFT"),
    ("What was Microsoft total revenue in Q1 FY2026?", "MSFT"),
]

print("Running test queries...\n")
print(f"{'Query':<55} {'Answer':<50} {'ms':>6}")
print("-" * 115)

for query, ticker in test_queries:
    result = answer_question_traced(query, ticker)
    print(f"{query[:54]:<55} {result['answer'][:49]:<50} {result['latency_ms']:>6}")

print("\n✓ All queries traced — check Langfuse dashboard")

Running test queries...

Query                                                   Answer                                                 ms
-------------------------------------------------------------------------------------------------------------------
What was Apple total net sales in Q2 2026?              Three Months Ended March 28, 2026, Apple total ne     869
What was Apple net income in Q2 2026?                   Three Months Ended March 28, 2026, Apple net inco     702
What was Apple gross margin in Q2 2026?                 Three Months Ended March 28, 2026. Apple gross ma     690
Why did Apple iPhone revenue increase in Q2 2026?       Higher net sales of Pro models during the Three M     571
What was Apple cash and cash equivalents?               Three Months Ended March 28, 2026, Apple cash and    5726
What was Apple operating income in Q2 2026?             Three Months Ended March 28, 2026. Apple operatin   18896
What was Microsoft net income in Q1 FY2026?             Not f